# Lab W4: Config, Seed, Logging, Checkpoint

## Pre-flight Checklist

> [!IMPORTANT]
> Konsep yang ditandai (§) merujuk ke `04_W4_Reproducibility_Experiment_Matrix.md`.

**Yang Anda butuhkan sebelum mulai:**
- Bab W4 sudah dibaca, terutama §3 Trace Result (empat hal yang direkam tiap run), §2 Training Terkontrol (seed variance dan Aturan 2σ), serta §1 Rancangan Penelitian (protokol dan pre-registration).
- Tugas jembatan W3 sudah disiapkan: satu diagnosis CIFAR-10 dari baseline W2 dan satu hipotesis ablation. Diagnosis itu dipakai untuk menulis `protocol.md`; notebook ini melatih infrastrukturnya dengan dataset toy.
- Familiar dengan `git status`, `git log`, `git diff`.

**Catatan kemandirian:** Notebook ini bisa dijalankan langsung di Google Colab tanpa import dari luar workspace notebook. Tidak ada file yang dibaca dari `../configs`, `../src`, atau `../experiments`. Config ditulis sebagai dictionary di notebook, dataset dibuat toy secara inline, dan seluruh keluaran ditulis ke folder lokal `lab4_outputs/`.

**Yang akan Anda hasilkan di akhir lab:**
- Verifikasi reproduksibilitas dari unit terkecil sampai satu run penuh: dua run seed sama → val_acc identik dalam 1e-4.
- Inspeksi metadata checkpoint (epoch, config, metrics, timestamp).
- Arsipkan folder output dengan nama yang jelas (`Exp-Ablasi/09-06-2026_09-40AM/`), lalu inisialisasi git repo dan push ke GitHub.
- Resume dari checkpoint: training dilanjutkan dari epoch N (bukan epoch 1).
- Plot perbandingan val_acc antar seed → estimasi seed variance untuk Aturan 2σ.

**Resource:**
- **Hardware:** CPU cukup; seluruh run memakai dataset toy kecil.
- **Estimasi waktu kerja:** 2-3 jam termasuk eksperimen, inspeksi, dan refleksi.

**Pendamping:** Bab W4 di `04_W4_Reproducibility_Experiment_Matrix.md`.

## Tujuan
1. Memverifikasi reproduksibilitas dari tensor sampai run: dua run dengan seed sama → hasil identik (± 1e-4).
2. Memeriksa metadata checkpoint secara lengkap (epoch, config, metrics, timestamp).
3. Memverifikasi resume dari checkpoint: lanjutkan training dari epoch N.
4. Membaca log training dan membandingkan variasi antar seed.

## Daftar Periksa
- [ ] Seed yang sama menghasilkan tensor acak identik (unit terkecil reproduksibilitas).
- [ ] Dua run dengan seed sama → val_acc identik hingga 4 desimal.
- [ ] Folder output diarsipkan dengan nama yang jelas dan bisa ditelusuri.
- [ ] Folder output diarsipkan dengan nama yang jelas dan bisa ditelusuri.
- [ ] Resume dari checkpoint: training dilanjutkan dari epoch N, bukan epoch 1.
- [ ] Paragraf refleksi tentang trade-off determinism vs kecepatan.

## Alur Lab (bottom-up)

Lab ini membangun reproducibility dari bukti terkecil ke audit eksperimen. Setiap langkah memakai objek yang sudah disiapkan di sel Setup, tanpa file luar:

1. **Satu tensor:** seed yang sama → dua tensor acak identik. Ini fondasi semua reproduksibilitas.
2. **Satu config dict:** baseline ditulis sebagai dictionary di notebook, bukan dibaca dari YAML repo.
3. **Satu mini-run:** latih toy dataset singkat → hasilkan `train.log`, `summary.json`, dan checkpoint di `lab4_outputs/`.
4. **Dua mini-run seed sama:** bandingkan `best_val_acc` untuk memeriksa determinism.
5. **Satu checkpoint:** buka metadata, config, dan metrics dari run sendiri.
6. **Satu arsip output:** download folder `lab4_outputs/` ke laptop, rename dengan pola `Exp-Ablasi/YYYY-MM-DD_HH-MM/`, lalu inisialisasi git repo dan push.
7. **Satu resume state:** muat model, optimizer, dan scheduler dari checkpoint.
8. **Beberapa seed:** plot variasi antar seed sebagai dasar Aturan 2σ.


## 1. Setup

Semua import, model, loss, config helper, dataset toy, training loop, dan helper git dikumpulkan di satu sel. Sel berikutnya hanya memakai objek yang sudah disiapkan di sini.

Keluaran ditulis ke `lab4_outputs/` relatif terhadap working directory notebook, bukan ke folder repo. Dengan begitu lab tetap mandiri di Google Colab.


In [ ]:
import json
import re
from datetime import datetime, timezone
from pathlib import Path

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset

# Keluaran ditulis relatif ke working directory notebook, bukan ke folder repo.
OUTPUT_DIR = Path("lab4_outputs").resolve()
OUTPUT_DIR.mkdir(exist_ok=True)


class FocalLoss(nn.Module):
    """Multi-class focal loss; gamma=0 dan alpha=None identik dengan CrossEntropyLoss."""

    def __init__(self, gamma=2.0, alpha=None, label_smoothing=0.0):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing
        if alpha is not None:
            self.register_buffer("alpha", torch.tensor(alpha, dtype=torch.float32))
        else:
            self.alpha = None

    def forward(self, logits, targets):
        ce = torch.nn.functional.cross_entropy(
            logits, targets, reduction="none", label_smoothing=self.label_smoothing
        )
        if self.gamma == 0.0 and self.alpha is None:
            return ce.mean()
        with torch.no_grad():
            probs = torch.nn.functional.softmax(logits, dim=1)
            pt = probs.gather(1, targets.unsqueeze(1)).squeeze(1).clamp_min(1e-8)
        loss = ((1.0 - pt) ** self.gamma) * ce
        if self.alpha is not None:
            loss = self.alpha.to(loss.device)[targets] * loss
        return loss.mean()


class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10, in_channels=3):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(in_channels, 32, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.2),
            nn.Linear(64, num_classes),
        )

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        return self.classifier(x)


def set_seed(seed, deterministic=True):
    """Kunci randomness di Python, NumPy, dan Torch (CPU + CUDA)."""
    import random
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    if deterministic:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False


def make_toy_images(seed=0, n_train=384, n_val=192, num_classes=10):
    """Dataset toy yang learnable: tiap kelas punya patch terang di lokasi berbeda.

    Generator di-seed terpisah sehingga data identik untuk seed yang sama. CNN bisa
    belajar pola spasial sederhana, jadi seed variance terlihat pada akurasi yang bermakna.
    """
    g = torch.Generator().manual_seed(seed)
    total = n_train + n_val
    labels = (torch.arange(total) % num_classes)
    labels = labels[torch.randperm(total, generator=g)]
    images = 0.20 * torch.randn(total, 3, 32, 32, generator=g)
    grid_positions = [(r, c) for r in [3, 11, 19, 25] for c in [3, 11, 19]]
    for i, label in enumerate(labels):
        y0, x0 = grid_positions[int(label) % len(grid_positions)]
        images[i, :, y0:y0 + 5, x0:x0 + 5] += 1.5
        images[i, int(label) % 3, :, :] += 0.05
    images = images.clamp(-1.0, 2.0)
    x_tr, y_tr = images[:n_train], labels[:n_train]
    x_va, y_va = images[n_train:], labels[n_train:]
    return (
        DataLoader(TensorDataset(x_tr, y_tr), batch_size=64, shuffle=True),
        DataLoader(TensorDataset(x_va, y_va), batch_size=64),
    )


def build_loss(cfg):
    name = cfg["name"].lower()
    if name in ("cross_entropy", "ce"):
        return nn.CrossEntropyLoss(label_smoothing=cfg.get("label_smoothing", 0.0))
    if name == "focal":
        return FocalLoss(gamma=cfg.get("gamma", 2.0), label_smoothing=cfg.get("label_smoothing", 0.0))
    raise ValueError(f"Loss belum dikenal: {name}")


def build_optimizer(params, cfg):
    name = cfg["name"].lower()
    if name == "sgd":
        return torch.optim.SGD(params, lr=cfg["lr"], momentum=cfg.get("momentum", 0.9),
                               weight_decay=cfg.get("weight_decay", 0.0), nesterov=cfg.get("nesterov", False))
    if name in ("adam", "adamw"):
        cls = torch.optim.AdamW if name == "adamw" else torch.optim.Adam
        return cls(params, lr=cfg["lr"], weight_decay=cfg.get("weight_decay", 0.0))
    raise ValueError(f"Optimizer belum dikenal: {name}")


def build_scheduler(optimizer, cfg, epochs):
    if not cfg or cfg.get("name", "none").lower() in ("none", "null"):
        return None
    if cfg["name"].lower() == "cosine":
        return torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(epochs, 1))
    if cfg["name"].lower() == "step":
        return torch.optim.lr_scheduler.StepLR(optimizer, step_size=cfg.get("step_size", 10),
                                               gamma=cfg.get("gamma", 0.1))
    raise ValueError(f"Scheduler belum dikenal: {cfg.get('name')}")


def apply_freeze(model, freeze_until):
    if freeze_until is None:
        return
    order = ["block1", "block2"]
    frozen = set()
    for name in order:
        frozen.add(name)
        if name == freeze_until:
            break
    for name, module in model.named_children():
        if name in frozen:
            for p in module.parameters():
                p.requires_grad = False


def evaluate(model, loader, loss_fn, device):
    model.eval()
    total, correct, loss_sum = 0, 0, 0.0
    with torch.no_grad():
        for x, y in loader:
            x = x.to(device)
            y = torch.as_tensor(y).long().view(-1).to(device)
            logits = model(x)
            loss_sum += loss_fn(logits, y).item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)
    return loss_sum / max(total, 1), correct / max(total, 1)


def run_mini(cfg, seed, suffix=""):
    """Satu run kecil end-to-end: train toy -> log + summary + checkpoint di lab4_outputs/.

    Mengembalikan dict ringkas. Seluruh keluaran ditulis ke folder lokal, tanpa file repo.
    """
    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    epochs = int(cfg["train"]["epochs"])
    train_loader, val_loader = make_toy_images(seed=seed, num_classes=cfg["model"].get("num_classes", 10))

    model = SimpleCNN(num_classes=cfg["model"].get("num_classes", 10)).to(device)
    apply_freeze(model, cfg["model"].get("freeze_until"))
    loss_fn = build_loss(cfg["loss"])
    optimizer = build_optimizer([p for p in model.parameters() if p.requires_grad], cfg["optim"])
    scheduler = build_scheduler(optimizer, cfg.get("scheduler"), epochs)

    out_dir = OUTPUT_DIR / f"{cfg['experiment_name']}_seed{seed}{suffix}"
    out_dir.mkdir(parents=True, exist_ok=True)
    log_lines = []
    last_epoch = 0
    for epoch in range(1, epochs + 1):
        model.train()
        run_loss = correct = total = 0
        for x, y in train_loader:
            x = x.to(device)
            y = torch.as_tensor(y).long().view(-1).to(device)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x)
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()
            run_loss += loss.item() * x.size(0)
            correct += (logits.argmax(1) == y).sum().item()
            total += x.size(0)
        if scheduler is not None:
            scheduler.step()
        train_loss, train_acc = run_loss / max(total, 1), correct / max(total, 1)
        val_loss, val_acc = evaluate(model, val_loader, loss_fn, device)
        last_epoch = epoch
        log_lines.append(
            f"epoch={epoch} train_loss={train_loss:.4f} train_acc={train_acc:.4f} "
            f"val_loss={val_loss:.4f} val_acc={val_acc:.4f}"
        )

    (out_dir / "train.log").write_text("\n".join(log_lines) + "\n", encoding="utf-8")
    summary = {"experiment_name": cfg["experiment_name"], "seed": seed,
               "best_val_acc": val_acc, "final_val_acc": val_acc, "epochs": epochs}
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")

    ckpt = {
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "config": cfg,
        "meta": {
            "epoch": last_epoch,
            "metrics": {"train_acc": train_acc, "val_acc": val_acc, "val_loss": val_loss},
            "timestamp": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "seed": seed,
        },
    }
    torch.save(ckpt, out_dir / "ckpt_last.pt")
    torch.save(ckpt, out_dir / "ckpt_best.pt")
    return {"out_dir": out_dir, "best_val_acc": val_acc, "epoch": last_epoch, "log": log_lines}


print("Output folder:", OUTPUT_DIR)
print("CUDA tersedia:", torch.cuda.is_available())

## 2. Mulai dari satu tensor: seed sama → hasil identik

Sebelum melatih apa pun, uji fondasi reproduksibilitas pada unit terkecil. `set_seed` yang sama harus menghasilkan tensor acak yang persis sama. Jika langkah ini gagal, seluruh klaim "dua run identik" di bawah ikut gugur.


In [ ]:
def draw_random(seed):
    set_seed(seed)
    weights = torch.randn(4, 4)
    labels = torch.randint(0, 10, (8,))
    return weights, labels


w1, y1 = draw_random(42)
w2, y2 = draw_random(42)
w_other, _ = draw_random(43)

same = torch.allclose(w1, w2) and torch.equal(y1, y2)
diff = torch.allclose(w1, w_other)

print("seed 42 vs 42 -> identik :", same)
print("seed 42 vs 43 -> identik :", diff)
print("max selisih seed 42 vs 43:", (w1 - w_other).abs().max().item())

assert same, "Seed sama menghasilkan tensor berbeda. Cek set_seed."
assert not diff, "Seed berbeda menghasilkan tensor identik. RNG tidak ter-seed."
print("OK: fondasi reproduksibilitas lulus.")


## 3. Definisikan satu baseline dict

Baseline dibuat sebagai dictionary langsung di notebook. Ini menggantikan file config eksternal (`configs/baseline.yaml`) agar lab tetap mandiri di Google Colab. Struktur dict-nya sengaja dibuat mirip YAML repo: `model`, `loss`, `optim`, `scheduler`, `train`.


In [ ]:
base_cfg = {
    "experiment_name": "lab4_baseline",
    "model": {"name": "simple_cnn", "num_classes": 10, "freeze_until": None},
    "loss": {"name": "cross_entropy", "label_smoothing": 0.0},
    "optim": {"name": "sgd", "lr": 0.05, "momentum": 0.9, "weight_decay": 5.0e-4, "nesterov": True},
    "scheduler": {"name": "cosine"},
    "train": {"epochs": 2},
}

print("Experiment :", base_cfg["experiment_name"])
print("Model      :", base_cfg["model"]["name"])
print("Loss       :", base_cfg["loss"]["name"])
print("Optimizer  :", base_cfg["optim"]["name"], "lr =", base_cfg["optim"]["lr"])
print("Epochs     :", base_cfg["train"]["epochs"])


## 4. Satu mini-run reproducible

Jalankan satu mini-run dari baseline dict. Run ini melatih dataset toy beberapa epoch lalu menulis tiga artefak ke `lab4_outputs/`: `train.log`, `summary.json`, dan checkpoint (`ckpt_best.pt` + `ckpt_last.pt`). Inilah objek yang akan diaudit di sel-sel berikutnya.


In [ ]:
r1 = run_mini(base_cfg, seed=42, suffix="_run1")

print("Folder run :", r1["out_dir"].relative_to(OUTPUT_DIR.parent))
print("Epoch akhir:", r1["epoch"])
print("best_val_acc:", f"{r1['best_val_acc']:.4f}")
print()
print("Isi folder:")
for f in sorted(r1["out_dir"].iterdir()):
    print(" -", f.name)
print()
print("train.log:")
print(r1["out_dir"].joinpath("train.log").read_text(encoding="utf-8").strip())


## 5. Dua mini-run seed sama → determinism

Jalankan run kedua dengan config dan seed yang persis sama. Dua run identik seharusnya menghasilkan `best_val_acc` yang sama hingga 4 desimal. Selisih besar berarti ada sumber non-determinisme (seed bocor, operasi GPU non-deterministic, atau urutan data berubah).


In [ ]:
r2 = run_mini(base_cfg, seed=42, suffix="_run2")

diff = abs(r1["best_val_acc"] - r2["best_val_acc"])
print(f"Run 1 val_acc: {r1['best_val_acc']:.6f}")
print(f"Run 2 val_acc: {r2['best_val_acc']:.6f}")
print(f"Selisih      : {diff:.6f}")

if diff < 1e-4:
    print("OK: reproduksibel untuk seed dan config yang sama.")
else:
    print("Tidak reproduksibel. Cek seed, deterministic flag, dan operasi GPU non-deterministic.")


## 6. Buka metadata checkpoint

Buka checkpoint dari run yang baru saja dibuat (bukan dari training repo eksternal). Checkpoint menyimpan lebih dari `model.state_dict()`: ada `config`, dan blok `meta` berisi `epoch`, `metrics`, dan `timestamp`. Tanpa metadata ini, sebuah checkpoint hanyalah setengah bukti.

In [ ]:
ckpt_path = r1["out_dir"] / "ckpt_best.pt"
ckpt = torch.load(ckpt_path, map_location="cpu")

print("Checkpoint:", ckpt_path.relative_to(OUTPUT_DIR.parent))
print()
print("=== Kunci checkpoint ===")
print(list(ckpt.keys()))
print()
print("=== Metadata (meta) ===")
for k, v in ckpt["meta"].items():
    print(f"{k:10s}: {v}")
print()
print("=== Ringkasan config ===")
for section in ["model", "loss", "optim", "scheduler", "train"]:
    print(f"{section:10s}: {ckpt['config'].get(section)}")


## 7. Arsipkan output dengan nama yang jelas

Folder `lab4_outputs/` di Colab tidak punya jejak git otomatis. Solusinya: download folder ke laptop, beri nama yang mengandung eksperimen dan waktu, lalu inisialisasi git repo dan push ke GitHub.

Langkah yang direkomendasikan:

1. Di Colab, klik kanan folder `lab4_outputs/` di panel Files lalu pilih **Download**.
2. Extract ke laptop, buat folder induk bernama `Exp-Ablasi/`.
3. Rename subfolder dengan pola `DD-MM-YYYY - HH_MMAM/` sesuai waktu run (contoh: `09-06-2026 - 09_40AM/`).
4. Di terminal laptop, jalankan perintah di sel bawah.

In [ ]:
# Perintah berikut dijalankan di TERMINAL LAPTOP setelah folder di-download dari Colab.
# Sesuaikan nama folder dan URL repo dengan milik Anda.

# Struktur folder yang direkomendasikan:
# Exp-Ablasi/
#   09-06-2026 - 09_40AM/
#     lab4_baseline_seed42_run1/
#       train.log
#       summary.json
#       ckpt_best.pt
#     plots/
#       lab4_seed_variance.png

# cd Exp-Ablasi
# git init
# git add .
# git commit -m "run awal - baseline seed 42, 43, 44"
# git remote add origin <URL-repo-GitHub-Anda>
# git push -u origin master

print("Ingat: tiap folder run punya nama seed + waktu yang unik.")
print("Setelah di-push, URL commit menjadi jejak permanen eksperimen ini.")

## 8. Verifikasi resume state dari checkpoint

Resume yang benar memuat kembali model, optimizer, dan scheduler, lalu melanjutkan dari epoch tersimpan - bukan memulai dari epoch 1. Sel ini memakai `ckpt_last.pt` dari run sendiri di `lab4_outputs/`.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt_path = r1["out_dir"] / "ckpt_last.pt"
ckpt = torch.load(ckpt_path, map_location=device)

resumed_epoch = ckpt["meta"]["epoch"]
total_epochs = ckpt["config"]["train"]["epochs"]

model = SimpleCNN(num_classes=ckpt["config"]["model"].get("num_classes", 10)).to(device)
model.load_state_dict(ckpt["model_state"])
apply_freeze(model, ckpt["config"]["model"].get("freeze_until"))

optimizer = build_optimizer([p for p in model.parameters() if p.requires_grad], ckpt["config"]["optim"])
optimizer.load_state_dict(ckpt["optimizer_state"])

scheduler = build_scheduler(optimizer, ckpt["config"].get("scheduler"), total_epochs)
if scheduler is not None and ckpt["scheduler_state"] is not None:
    scheduler.load_state_dict(ckpt["scheduler_state"])

print(f"Checkpoint dimuat: {ckpt_path.relative_to(OUTPUT_DIR.parent)}")
print(f"Epoch tersimpan  : {resumed_epoch} dari total {total_epochs}")
if resumed_epoch >= total_epochs:
    print("Checkpoint sudah di epoch terakhir untuk run ini.")
    print("Pada run lebih panjang, resume melanjutkan dari epoch tersimpan + 1 sampai total_epochs.")
else:
    print(f"Siap melanjutkan dari epoch {resumed_epoch + 1} sampai {total_epochs}.")
print(f"LR saat ini      : {optimizer.param_groups[0]['lr']:.6f}")


## 9. Variasi antar seed → Aturan 2σ

Sekarang ukur seed variance. Jalankan baseline dengan beberapa seed berbeda, lalu plot val_acc akhirnya. Sebaran ini adalah noise dasar: sebuah perbedaan antar konfigurasi baru dianggap bermakna kalau lebih besar dari kira-kira 2σ sebaran seed ini (§2 Training Terkontrol).

In [ ]:
def parse_log(log_path):
    pattern = re.compile(
        r"epoch=(\d+) train_loss=([\d.]+) train_acc=([\d.]+) val_loss=([\d.]+) val_acc=([\d.]+)"
    )
    epochs, val_acc = [], []
    for line in log_path.read_text(encoding="utf-8").splitlines():
        m = pattern.search(line)
        if m:
            epochs.append(int(m.group(1)))
            val_acc.append(float(m.group(5)))
    return epochs, val_acc


seeds = [42, 123, 2024]
seed_runs = {s: run_mini(base_cfg, seed=s, suffix="_seedvar") for s in seeds}

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["steelblue", "tomato", "seagreen"]
final_accs = []
for i, s in enumerate(seeds):
    epochs, val_acc = parse_log(seed_runs[s]["out_dir"] / "train.log")
    ax.plot(epochs, [v * 100 for v in val_acc], marker="o", color=colors[i % len(colors)], label=f"seed={s}")
    final_accs.append(val_acc[-1])

ax.set_xlabel("Epoch")
ax.set_ylabel("Val Accuracy (%)")
ax.set_title("Lab 4: variasi val_acc antar seed (config identik)")
ax.legend()
ax.grid(alpha=0.3)
plot_dir = OUTPUT_DIR / "plots"
plot_dir.mkdir(exist_ok=True)
plt.tight_layout()
plt.savefig(plot_dir / "lab4_seed_variance.png", dpi=120, bbox_inches="tight")
plt.show()

mean = np.mean(final_accs) * 100
std = np.std(final_accs) * 100
print(f"Final val_acc per seed: {[f'{a*100:.2f}%' for a in final_accs]}")
print(f"mean = {mean:.2f}%, std = {std:.2f}%")
print(f"Aturan 2σ: perbedaan antar konfigurasi baru bermakna jika > {2 * std:.2f}% (perkiraan kasar).")


## 10. Refleksi

1. Jika kamu menemukan bug dua minggu kemudian dan harus memahami ulang eksperimen ini, apa **tiga minimum** yang kamu butuhkan dari folder eksperimen - bukan dari ingatanmu?

2. `cudnn.deterministic=True` memperlambat training ~10-20%. Situasi mana yang membuatmu menerima ketidakdeterminisan demi kecepatan - dan situasi mana yang tidak?

3. Bayangkan kamu menemukan folder output lama tanpa timestamp dan tanpa git history. Apa langkah yang akan kamu ambil sebelum menggunakan hasilnya di paper?


### Jawaban Refleksi

**1. Tiga minimum dari folder eksperimen:**
> *[tulis di sini]*

**2. Kapan menerima/menolak non-determinism:**
> *[tulis di sini]*

**3. Folder output tanpa timestamp dan tanpa git:**
> *[tulis di sini]*
